# The competition's evaluator loop on a Colab GPU (spec 028 AI9, #687)

**Runtime -> Change runtime type -> A100 GPU** (an L4 also works). Then **Runtime -> Run all**.
With Colab Pro+ (background execution) the tab can be closed while it runs; how long a session may last is Colab's to say (its FAQ), not this page's.

**Once, before Run all** (already done if you ran `colab_agent.ipynb` before): the key icon on the left (Secrets), two
secrets with *Notebook access* on: `KAGGLE_USERNAME` = `jcblearning`, and `KAGGLE_API_TOKEN` = your Kaggle API token
(kaggle.com -> Settings -> API -> Generate New Token). Never paste the token into a cell.

What it does: installs the model server and the agent framework at the versions the Kaggle kernels used, puts the
competition's wheels, the model and one private task bundle (no fix, no tests) where Kaggle would mount them (for a
batch of the competition's development tasks, the competition's files for those tasks alone, fetched with the token
below, #780), and runs
the repository's own kernel file unchanged. The log and the output go to a private dataset
`jcblearning/htlnlp-colab-official-<label>` when it ends, however it ends, so nothing needs copying from this page.
Which runs it makes is read from `kaggle/colab_official_run.json` on the branch. **One tap is enough:** after the runs it started with, it keeps reading that list (every minute, for up to two and a half hours after each run) and makes any run added to it, with this branch's helper as it then is. So a full arm, whose plan is registered only after the test run has been read, starts without a second tap. When the list says stop, or names nothing new in that time, it gives the GPU back.


In [ ]:
import json, subprocess, time, urllib.request
info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True).stdout.strip()
name, mem = [p.strip() for p in info.split(",")]
gb = float(mem) / 1024
print(f"{name}  {gb:.1f} GB")
# The 31B needs Colab's A100 40GB (as every c31be8* session); E4B fits an L4 (about 10 GB). The queued runs decide.
url = ("https://raw.githubusercontent.com/healthtransformationlab/health-transformation-lab-nlp/refs/heads/"
       f"data/849-028-colab-public-bundle/kaggle/colab_official_run.json?t={int(time.time())}")
runs = json.loads(urllib.request.urlopen(url, timeout=60).read().decode()).get("runs", [])
need = 35 if any("31b" in str(r.get("model", "31b")).lower() for r in runs) else 20
if gb < need:
    raise SystemExit(
        f"STOP: this GPU has {gb:.0f} GB and the queued runs need about {need + 5} GB. "
        "Runtime -> Change runtime type -> A100 GPU, then Run all again.")
print(f"OK: {gb:.0f} GB is enough for the queued runs ({', '.join(r['label'] for r in runs)}). Continuing.")


In [ ]:
#@title Tap the round play button on the left, or Runtime -> Run all. The code is hidden so a stray tap cannot change it. { display-mode: "form" }
import json, os, time, types, urllib.request
from google.colab import userdata

def secret(name, read, default=None):   # #827: only the token must be a Colab secret; the username is not one
    try:
        value, why = read(), "empty"
    except Exception as e:   # SecretNotFoundError, or NotebookAccessError when it isn't shared with this notebook
        value, why = None, type(e).__name__
    if value:
        return value
    if default is not None:
        print(f"Colab secret {name}: {why}, so {default} is used")
        return default
    raise SystemExit(f"Colab secret {name}: {why}. Key icon on the left -> Add new secret {name} (your Kaggle API "
                     f"token), turn Notebook access on, then Runtime -> Run all again.")

os.environ["KAGGLE_USERNAME"] = secret("KAGGLE_USERNAME", lambda: userdata.get("KAGGLE_USERNAME"), "jcblearning")
os.environ["KAGGLE_API_TOKEN"] = secret("KAGGLE_API_TOKEN", lambda: userdata.get("KAGGLE_API_TOKEN"))

REPO = "healthtransformationlab/health-transformation-lab-nlp"
BRANCH = "data/849-028-colab-public-bundle"   # the runs of #687 were made from feat/687-028-colab-paired-run

def get(name):   # the branch's own file; the query keeps a cached copy from being served after a fix
    url = f"https://raw.githubusercontent.com/{REPO}/refs/heads/{BRANCH}/kaggle/{name}?t={int(time.time())}"
    return urllib.request.urlopen(url, timeout=60).read().decode()

co = types.ModuleType("colab_official")
exec(compile(get("colab_official.py"), "colab_official.py", "exec"), co.__dict__)
plan = json.loads(get("colab_official_run.json"))
print("runs:", [r["label"] for r in plan["runs"]])
for r in plan["runs"]:
    code = co.main(r["label"], r["bundle"], kernel_dataset=r.get("kernel_dataset"), kernel_url=r.get("kernel_url"))
    print(r["label"], "finished, kernel exit", code)
print("ALL DONE. Nothing needs copying: the log and results are on Kaggle.")
